In [1]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/nellyoa/enose-coffee"
    REPO_NAME = "enose-coffee"

    # Clone repositori
    if not os.path.exists(REPO_NAME):
        print(f"Download the Repository in GitHub...")
        !git clone {REPO_URL}

    CSV_PATH = os.path.join("/content", REPO_NAME, "data", "Rasio_Gabungan_Semua_Grade.csv")
    print(f"Run in Colab. Path dataset: {CSV_PATH}")

else:

    CSV_PATH = os.path.join("..", "data", "Rasio_Gabungan_Semua_Grade.csv")
    print(f"Run in Local. Path dataset: {CSV_PATH}")


Download the Repository in GitHub...
Cloning into 'enose-coffee'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 39 (delta 15), reused 29 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 15.40 MiB | 7.51 MiB/s, done.
remote: Total 39 (delta 15), reused 29 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 15.40 MiB | 7.51 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Resolving deltas: 100% (15/15), done.
Run in Colab. Path dataset: /content/enose-coffee/data/Rasio_Gabungan_Semua_Grade.csv
Run in Colab. Path dataset: /content/enose-coffee/data/Rasio_Gabungan_Semua_Grade.csv


In [2]:
# %% 0. Imports and configuration
import os, sys, json, math, time, random, platform, re, zipfile, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy import stats as sps
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

# ---- user settings ---------------------------------------------------------------------------------------------
WORK = "/content/enose_repro"                                       # everything is written here
QUICK = False            # True: 3 epochs, 1 seed, 1 repeat, few models -> smoke test only (numbers are meaningless)
RUN = dict(build=True, p1=True, p1_legacy=True, p2=True, p3=True, baselines=True, normalisation=True,
           drift=True, stats=True, figures=True, params=True,
           segments=False,   # segment ablation (early/mid/late window): +12 runs sets, slow
           latency=False,    # CPU latency; meaningful only on an otherwise idle CPU
           extra=True,       # revision controls E1-E4 (matched data, random 300/100/100 control, capacity-matched models, TCN, Transformer): several hours
           stats_v2=True)    # revised statistics (cluster bootstrap over 25 batches, Holm family, +-5 point equivalence) and Tables 4-7
ALL_MODELS = ["cnn_flatten", "cnn_lstm", "cnn_bigru", "cnn_gru", "cnn_gap"]
SEEDS = [1, 2, 3]                      # model-initialisation seeds (P1 and P3)
SPLIT_SEED = 42                        # random_state of the original split
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 32, 100, 12   # Adam lr, batch size, epochs, early stopping on val_loss
GPU = bool(tf.config.list_physical_devices("GPU"))
DETERMINISTIC = not GPU                # op determinism can raise errors on some GPU ops; off on GPU by default

DATA = f"{WORK}/data"; RUNS = f"{WORK}/results/runs"; TABLES = f"{WORK}/results/tables"; FIGS = f"{WORK}/results/figures"
for d in (DATA, RUNS, TABLES, FIGS): os.makedirs(d, exist_ok=True)
NPZ = f"{DATA}/enose_coffee.npz"
if QUICK:
    SEEDS, MAX_EPOCHS = [1], 3
print("TF", tf.__version__, "| GPU:", GPU, "| QUICK:", QUICK, "| work dir:", WORK)

TF 2.21.0 | GPU: True | QUICK: False | work dir: /content/enose_repro


In [3]:
# %% 1. (Colab only) upload the CSV if it is not on disk
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        up = files.upload()                                 # choose the .csv or the .zip
        CSV_PATH = os.path.join(os.getcwd(), list(up)[0])
    except ImportError:
        raise SystemExit(f"CSV not found: {CSV_PATH}. Set CSV_PATH.")
if CSV_PATH.lower().endswith(".zip"):
    with zipfile.ZipFile(CSV_PATH) as z:
        name = [n for n in z.namelist() if n.lower().endswith(".csv") and "__MACOSX" not in n][0]
        z.extract(name, DATA); CSV_PATH = os.path.join(DATA, name)
print("CSV:", CSV_PATH)

CSV: /content/enose-coffee/data/Rasio_Gabungan_Semua_Grade.csv


In [4]:
# %% 2. Build the array: first 700 s of each 1000 s cycle, 8 MOS channels; batch_id = solution 0..4 within a grade
def build_npz(csv, out, grade_col="Grade", sample_col="Original_Sample", cycle_col="Siklus", n_steps=700, cycle_len=1000):
    df = pd.read_csv(csv)
    if df[sample_col].dtype == object:
        df[sample_col] = df[sample_col].astype(str).str.extract(r"(\d+)")[0].astype(int)      # "Sample 3" -> 3
    sensors = ["MQ2", "MQ3", "MQ4", "MQ7", "MQ8", "MQ135", "MQ137", "TGS"]
    assert all(c in df.columns for c in sensors), f"expected sensor columns {sensors}, got {list(df.columns)}"
    gmap = {g: i for i, g in enumerate(sorted(df[grade_col].unique()))}
    X, y, batch, run = [], [], [], []
    for (g, s), gs in df.groupby([grade_col, sample_col], sort=True):
        for ci, c in enumerate(sorted(gs[cycle_col].unique())):
            cyc = gs[gs[cycle_col] == c]
            if len(cyc) < n_steps: raise SystemExit(f"grade {g} sample {s} cycle {c}: only {len(cyc)} rows")
            X.append(cyc[sensors].to_numpy(np.float32)[:n_steps]); y.append(gmap[g]); batch.append(s); run.append(ci)
    X, y, run = np.stack(X), np.array(y), np.array(run)
    b_raw, batch_id = np.array(batch), np.zeros(len(y), int)
    for g in np.unique(y):
        for i, v in enumerate(sorted(set(b_raw[y == g]))): batch_id[(y == g) & (b_raw == v)] = i
    tab = pd.crosstab(y, batch_id); print("X", X.shape, "NaN:", int(np.isnan(X).sum())); print(tab)
    assert X.shape == (500, n_steps, 8) and (tab.to_numpy() == 20).all(), "data do not match 5 grades x 5 batches x 20 cycles"
    np.savez_compressed(out, X=X, y=y, batch_id=batch_id, run_order=run)
if RUN["build"] or not os.path.exists(NPZ):
    build_npz(CSV_PATH, NPZ)

X (500, 700, 8) NaN: 0
col_0   0   1   2   3   4
row_0                    
0      20  20  20  20  20
1      20  20  20  20  20
2      20  20  20  20  20
3      20  20  20  20  20
4      20  20  20  20  20


In [5]:
# %% 3. Data, models, splits, training (identical to the reproducibility package)
def load_dataset(path):
    d = np.load(path); X, y = d["X"].astype(np.float32), d["y"].astype(int)
    meta = pd.DataFrame({"sample_id": np.arange(len(y)), "grade": y, "batch_id": d["batch_id"].astype(int), "run_order": d["run_order"].astype(int)})
    meta["group"] = meta["grade"] * 100 + meta["batch_id"]          # one id per physical solution
    return X, y, meta
def fit_standardizer(Xt):
    flat = Xt.reshape(-1, Xt.shape[-1]); mu, sd = flat.mean(0), flat.std(0); sd[sd == 0] = 1.0
    return mu.astype(np.float32), sd.astype(np.float32)
def apply_standardizer(X, mu, sd): return ((X - mu) / sd).astype(np.float32)

DROPOUTS = {"cnn_flatten": (0.2, 0.0, 0.2), "cnn_lstm": (0.0, 0.4, 0.3), "cnn_bigru": (0.0, 0.4, 0.3),
            "cnn_gru": (0.0, 0.4, 0.3), "cnn_gap": (0.0, 0.4, 0.3)}     # (after conv2, after core layer, after Dense)
def _conv(x, f, k):
    x = layers.Conv1D(f, k, padding="same", activation="relu")(x); x = layers.BatchNormalization()(x); return layers.MaxPooling1D(2)(x)
def build_model(name, input_shape=(700, 8), n_classes=5, dense_units=64):
    d_conv, d_core, d_dense = DROPOUTS[name]
    inp = keras.Input(shape=input_shape); x = _conv(inp, 32, 7); x = _conv(x, 64, 5)
    if d_conv: x = layers.Dropout(d_conv)(x)
    x = {"cnn_flatten": lambda x: layers.Flatten()(x), "cnn_lstm": lambda x: layers.LSTM(64)(x),
         "cnn_bigru": lambda x: layers.Bidirectional(layers.GRU(64))(x), "cnn_gru": lambda x: layers.GRU(64)(x),
         "cnn_gap": lambda x: layers.GlobalAveragePooling1D()(x)}[name](x)
    if d_core: x = layers.Dropout(d_core)(x)
    x = layers.Dense(dense_units, activation="relu")(x); x = layers.Dropout(d_dense)(x)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x), name=name)
def count_params(m):
    tr = sum(int(w.numpy().size) for w in m.trainable_weights); nt = sum(int(w.numpy().size) for w in m.non_trainable_weights)
    return tr, nt, tr + nt

def original_protocol_split(y, seed=42):
    y_cat = np.eye(int(y.max()) + 1)[y]; idx = np.arange(len(y))        # the notebook stratifies on the one-hot matrix
    tr, tmp = train_test_split(idx, test_size=0.25, random_state=seed, stratify=y_cat)
    val, te = train_test_split(tmp, test_size=0.5, random_state=seed, stratify=y_cat[tmp])
    return [{"name": f"original_seed{seed}", "train": tr, "val": val, "test": te}]
def repeated_stratified_kfold(y, n_splits=5, n_repeats=5, val_fraction=0.1, seed=0):
    folds = []
    for i, (tr_all, te) in enumerate(RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed).split(np.zeros(len(y)), y)):
        tr, val = train_test_split(tr_all, test_size=val_fraction, stratify=y[tr_all], random_state=seed + i)
        folds.append({"name": f"rskf_r{i // n_splits}_f{i % n_splits}", "train": tr, "val": val, "test": te})
    return folds
def batch_grouped_folds(meta, n_batches=5):
    b = meta["batch_id"].to_numpy(); folds = []
    for k in range(n_batches):                       # test = solution k of every grade, val = solution k+1, train = other three
        folds.append({"name": f"grouped_test_batch{k}", "train": np.where((b != k) & (b != (k + 1) % n_batches))[0],
                      "val": np.where(b == (k + 1) % n_batches)[0], "test": np.where(b == k)[0]})
    return folds
def leakage_report(meta, f):
    g_tr, g_te = set(meta.loc[f["train"], "group"]), set(meta.loc[f["test"], "group"]); sh = g_tr & g_te
    return {"fold": f["name"], "n_test": int(len(f["test"])), "batches_in_test": len(g_te), "batches_shared_with_train": len(sh),
            "test_samples_whose_batch_is_also_in_train": int(meta.loc[f["test"], "group"].isin(sh).sum())}
def make_folds(protocol, y, meta, n_repeats=2):
    return {"original": lambda: original_protocol_split(y, SPLIT_SEED),
            "repeated_kfold": lambda: repeated_stratified_kfold(y, n_repeats=n_repeats, seed=0),
            "batch_grouped": lambda: batch_grouped_folds(meta)}[protocol]()

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
    if DETERMINISTIC:
        try: tf.config.experimental.enable_op_determinism()
        except Exception: pass

def run_protocol(name, protocol, models, seeds=None, scaler="train_only", segment=None, n_repeats=2):
    """Train/evaluate `models` under `protocol`; writes RUNS/<name>/{predictions,metrics_per_fold,summary,...}.csv/json"""
    out = f"{RUNS}/{name}"; os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/predictions.csv"):
        print("[skip] exists:", name); return
    X, y, meta = load_dataset(NPZ)
    if segment: a_, b_ = (int(v) for v in segment.split(":")); X = X[:, a_:b_]
    folds = make_folds(protocol, y, meta, 1 if QUICK else n_repeats)
    if QUICK: folds = folds[:2]
    json.dump([leakage_report(meta, f) for f in folds], open(f"{out}/leakage_report.json", "w"), indent=2)
    json.dump([{k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in f.items()} for f in folds], open(f"{out}/splits.json", "w"))
    seeds = seeds or SEEDS; models = models[:2] if QUICK else models
    rows, mrows, pcs = [], [], {}
    for f in folds:
        tr, va, te = f["train"], f["val"], f["test"]
        mu, sd = fit_standardizer(X if scaler == "global_legacy" else X[tr])
        Xtr, Xva, Xte = (apply_standardizer(X[i], mu, sd) for i in (tr, va, te))
        ytr, yva = (keras.utils.to_categorical(y[i], 5) for i in (tr, va))
        for m, s in [(m, s) for m in models for s in seeds]:
            set_seed(s); mod = build_model(m, X.shape[1:], 5)
            pcs[m] = dict(zip(("trainable", "non_trainable", "total"), count_params(mod)))
            mod.compile(optimizer=keras.optimizers.Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
            h = mod.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_EPOCHS, batch_size=BATCH, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)])
            pd.DataFrame(h.history).assign(epoch=lambda d: d.index + 1).to_csv(f"{out}/history_{m}_{f['name']}_s{s}.csv", index=False)
            prob = mod.predict(Xte, verbose=0); pred = prob.argmax(1)
            p, r, f1, _ = precision_recall_fscore_support(y[te], pred, average="macro", zero_division=0)
            mrows.append(dict(model=m, seed=s, fold=f["name"], n_test=len(te), epochs_run=len(h.history["loss"]), accuracy=float((pred == y[te]).mean()),
                              precision_macro=p, recall_macro=r, f1_macro=f1))
            for j, i in enumerate(te):
                row = dict(model=m, seed=s, fold=f["name"], sample_id=int(i), grade=int(y[i]), batch_id=int(meta.batch_id[i]), y_true=int(y[i]), y_pred=int(pred[j]))
                row.update({f"p_grade{c + 1}": float(prob[j, c]) for c in range(5)}); rows.append(row)
            print(f"[{name}|{f['name']}] {m:<12} seed={s} acc={mrows[-1]['accuracy']:.4f} epochs={mrows[-1]['epochs_run']}", flush=True)
            keras.backend.clear_session()
    preds, met = pd.DataFrame(rows), pd.DataFrame(mrows)
    preds.to_csv(f"{out}/predictions.csv", index=False); met.to_csv(f"{out}/metrics_per_fold.csv", index=False)
    met.groupby("model")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].agg(["mean", "std"]).to_csv(f"{out}/summary.csv")
    json.dump(pcs, open(f"{out}/param_counts.json", "w"), indent=2)
    for m in models:
        sub = preds[(preds.model == m) & (preds.seed == seeds[0])]
        pd.DataFrame(confusion_matrix(sub.y_true, sub.y_pred, labels=list(range(5))), index=[f"true_G{i + 1}" for i in range(5)],
                     columns=[f"pred_G{i + 1}" for i in range(5)]).to_csv(f"{out}/confusion_{m}.csv")
    print(met.groupby("model").accuracy.agg(["mean", "std"]).round(4))

In [6]:
# %% 4. Parameter counts (the paper reports 729,509 / 49,829 / 70,821 trainable)
if RUN["params"]:
    pc = {m: dict(zip(("trainable", "non_trainable", "total"), count_params(build_model(m)))) for m in ALL_MODELS}
    json.dump(pc, open(f"{TABLES}/param_counts.json", "w"), indent=2); print(pd.DataFrame(pc).T)

             trainable  non_trainable   total
cnn_flatten     729509            192  729701
cnn_lstm         49829            192   50021
cnn_bigru        70821            192   71013
cnn_gru          41765            192   41957
cnn_gap          16805            192   16997


In [7]:
# %% 5. Leakage-aware experiments (names match the statistics step below)
M5 = ALL_MODELS
if RUN["p1"]:        run_protocol("orig_train_only", "original", M5, scaler="train_only")                  # P1, 3 seeds
if RUN["p1_legacy"]: run_protocol("orig_global_legacy", "original", M5, scaler="global_legacy")           # P1 with the old scaler
if RUN["p2"]:
    run_protocol("rkf", "repeated_kfold", ["cnn_flatten", "cnn_lstm", "cnn_bigru"], seeds=[1], n_repeats=2)  # P2, 2 repeats x 5 folds
    run_protocol("rkf2", "repeated_kfold", ["cnn_gru", "cnn_gap"], seeds=[1], n_repeats=2)
if RUN["p3"]:        run_protocol("grouped", "batch_grouped", M5)                                          # P3, 3 seeds
if RUN["segments"]:
    for seg in ("0:200", "200:500", "500:700"):
        for pr in ("original", "batch_grouped"):
            run_protocol(f"seg_{seg.replace(':', '_')}_{pr}", pr, ["cnn_gap", "cnn_bigru"], segment=seg)

[orig_train_only|original_seed42] cnn_flatten  seed=1 acc=0.8889 epochs=25
[orig_train_only|original_seed42] cnn_flatten  seed=2 acc=0.9206 epochs=22


[orig_train_only|original_seed42] cnn_flatten  seed=3 acc=0.9683 epochs=28
[orig_train_only|original_seed42] cnn_lstm     seed=1 acc=0.9524 epochs=31
[orig_train_only|original_seed42] cnn_lstm     seed=2 acc=0.9365 epochs=29
[orig_train_only|original_seed42] cnn_lstm     seed=3 acc=0.9683 epochs=32
[orig_train_only|original_seed42] cnn_bigru    seed=1 acc=0.9206 epochs=31
[orig_train_only|original_seed42] cnn_bigru    seed=2 acc=0.9841 epochs=38
[orig_train_only|original_seed42] cnn_bigru    seed=3 acc=0.9524 epochs=39
[orig_train_only|original_seed42] cnn_gru      seed=1 acc=0.9683 epochs=39
[orig_train_only|original_seed42] cnn_gru      seed=2 acc=0.9524 epochs=51
[orig_train_only|original_seed42] cnn_gru      seed=3 acc=0.9524 epochs=36
[orig_train_only|original_seed42] cnn_gap      seed=1 acc=0.9683 epochs=52
[orig_train_only|original_seed42] cnn_gap      seed=2 acc=0.9841 epochs=55
[orig_train_only|original_seed42] cnn_gap      seed=3 acc=0.9841 epochs=51
               mean     s

In [8]:
# %% 6. Hand-crafted baselines on exactly the same indices, and the normalisation study
def hc_features(X):
    t = np.arange(X.shape[1]);
    trap = getattr(np, "trapezoid", None) or np.trapz
    sl = ((t - t.mean())[None, :, None] * (X - X.mean(1, keepdims=True))).sum(1) / ((t - t.mean()) ** 2).sum()
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1), X.argmax(1) / X.shape[1], sl, trap(X, axis=1) / X.shape[1],
                           X[:, :100].mean(1), X[:, -100:].mean(1), np.abs(np.diff(X, axis=1)).max(1)], 1)
def run_baselines(protocol, outname, seeds=(1, 2, 3), n_repeats=2):
    out = f"{RUNS}/{outname}"; os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/predictions.csv"): print("[skip]", outname); return
    X, y, meta = load_dataset(NPZ); F = hc_features(X); folds = make_folds(protocol, y, meta, n_repeats)
    seeds = seeds[:1] if QUICK else seeds
    models = {"RF": lambda s: RandomForestClassifier(300, random_state=s, n_jobs=-1),
              "SVM-RBF": lambda s: make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", random_state=s)),
              "LogReg": lambda s: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=s)),
              "MLP": lambda s: make_pipeline(StandardScaler(), MLPClassifier((64,), max_iter=500, early_stopping=True, random_state=s))}
    preds = []
    for f in folds:
        tr = np.concatenate([f["train"], f["val"]]) if protocol == "original" else f["train"]
        for m, mk in models.items():
            for s in seeds:
                p = mk(s).fit(F[tr], y[tr]).predict(F[f["test"]])
                preds += [dict(model=m, seed=s, fold=f["name"], sample_id=int(i), grade=int(y[i]), batch_id=int(meta.batch_id.values[i]), y_true=int(y[i]), y_pred=int(pi)) for i, pi in zip(f["test"], p)]
    P = pd.DataFrame(preds); P.to_csv(f"{out}/predictions.csv", index=False)
    print(outname, P.assign(ok=P.y_true == P.y_pred).groupby("model").ok.mean().round(3).to_dict())
if RUN["baselines"]:
    run_baselines("original", "base_original");
    run_baselines("repeated_kfold", "base_rkf");
    run_baselines("batch_grouped", "base_batch_grouped")

if RUN["normalisation"]:
    X, y, meta = load_dataset(NPZ)
    V = {"raw": X, "baseline10": X / X[:, :10].mean(1, keepdims=True), "zscore_rec": (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-8),
         "subtract_start": X - X[:, :10].mean(1, keepdims=True)}
    rows = []
    for v, Xv in V.items():
        F = hc_features(Xv)
        for f in batch_grouped_folds(meta):
            for m, clf in {"SVM-RBF": make_pipeline(StandardScaler(), SVC(C=10)), "RF": RandomForestClassifier(300, random_state=0, n_jobs=-1)}.items():
                clf.fit(F[f["train"]], y[f["train"]]); rows.append(dict(variant=v, model=m, fold=f["name"], acc=float((clf.predict(F[f["test"]]) == y[f["test"]]).mean())))
    r = pd.DataFrame(rows); r.to_csv(f"{TABLES}/normalisation_study.csv", index=False)
    print("normalisation study (batch-grouped):"); print(r.groupby(["variant", "model"]).acc.agg(["mean", "std"]).round(3))

base_original {'LogReg': 0.873, 'MLP': 0.862, 'RF': 0.974, 'SVM-RBF': 0.984}
base_rkf {'LogReg': 0.844, 'MLP': 0.775, 'RF': 0.921, 'SVM-RBF': 0.952}
base_batch_grouped {'LogReg': 0.456, 'MLP': 0.423, 'RF': 0.495, 'SVM-RBF': 0.522}
normalisation study (batch-grouped):
                         mean    std
variant        model                
baseline10     RF       0.506  0.142
               SVM-RBF  0.498  0.124
raw            RF       0.502  0.131
               SVM-RBF  0.522  0.094
subtract_start RF       0.512  0.052
               SVM-RBF  0.536  0.094
zscore_rec     RF       0.514  0.121
               SVM-RBF  0.450  0.140


In [9]:
# %% 7. Within-batch drift check: Spearman correlation between cycle number and response
if RUN["drift"]:
    d = np.load(NPZ); X, y, b, r_ = d["X"], d["y"], d["batch_id"], d["run_order"]; rows = []
    for g in np.unique(y):
        for k in np.unique(b[y == g]):
            m = (y == g) & (b == k); order = np.argsort(r_[m])
            for stat, F in {"mean": X[m][order].mean(1), "max": X[m][order].max(1)}.items():
                for ch in range(8): rows.append(dict(grade=g + 1, batch=k + 1, channel=ch + 1, stat=stat, rho=sps.spearmanr(np.arange(len(F)), F[:, ch])[0]))
    res = pd.DataFrame(rows); res.to_csv(f"{TABLES}/drift_check.csv", index=False)
    print(res.assign(abs_rho=res.rho.abs()).pivot_table(index="channel", columns="stat", values="abs_rho").round(2))
    print("share of (batch, channel) pairs with |rho| > 0.5:", round(float((res.rho.abs() > 0.5).mean()), 2))

stat      max  mean
channel            
1        0.50  0.64
2        0.38  0.72
3        0.40  0.66
4        0.60  0.68
5        0.55  0.65
6        0.49  0.82
7        0.47  0.69
8        0.48  0.67
share of (batch, channel) pairs with |rho| > 0.5: 0.66


In [10]:
# %% 8. Statistics: Table 5, pairwise tests, scaler effect, recall by grade
def wilson_ci(k, n, alpha=0.05):
    z = sps.norm.ppf(1 - alpha / 2); p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den; return c - h, c + h
def mcnemar_exact(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool); n01, n10 = int(np.sum(a & ~b)), int(np.sum(~a & b)); n = n01 + n10
    return {"A_right_B_wrong": n01, "A_wrong_B_right": n10, "p_value": 1.0 if n == 0 else float(min(1.0, 2 * sps.binom.cdf(min(n01, n10), n, 0.5)))}
def nadeau_bengio_ttest(a, b, n_train, n_test):
    d = np.asarray(a, float) - np.asarray(b, float); k = len(d); var = d.var(ddof=1)
    if var == 0: return {"mean_diff": float(d.mean()), "t": float("nan"), "p_value": 1.0 if d.mean() == 0 else 0.0}
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var); return {"mean_diff": float(d.mean()), "t": float(t), "p_value": float(2 * sps.t.sf(abs(t), df=k - 1))}

def compute_stats():
    DL = ["cnn_flatten", "cnn_lstm", "cnn_gru", "cnn_bigru", "cnn_gap"]; HC = ["SVM-RBF", "RF", "LogReg", "MLP"]
    def load(d):
        p = pd.read_csv(f"{RUNS}/{d}/predictions.csv"); p["ok"] = (p.y_true == p.y_pred).astype(int); return p
    ex = lambda d: os.path.exists(f"{RUNS}/{d}/predictions.csv")
    P = {"P1": pd.concat([load(d) for d in ("orig_train_only", "base_original") if ex(d)], ignore_index=True),
         "P2": pd.concat([load(d) for d in ("rkf", "rkf2", "base_rkf") if ex(d)], ignore_index=True),
         "P3": pd.concat([load(d) for d in ("grouped", "base_batch_grouped") if ex(d)], ignore_index=True)}
    rng = np.random.default_rng(0); rows = []
    for k, p in P.items():
        for m in DL + HC:
            g = p[p.model == m]
            if g.empty: continue
            seeds = g.groupby("seed").ok.mean(); r = dict(protocol=k, model=m, n_seeds=len(seeds), acc_mean=seeds.mean(), acc_sd_seeds=seeds.std(ddof=1) if len(seeds) > 1 else np.nan)
            if k == "P3":      # cluster bootstrap over the 25 solutions
                u = g.groupby(["grade", "batch_id", "seed"]).ok.mean().groupby(level=[0, 1]).mean().values
                r["ci_lo"], r["ci_hi"] = np.quantile(rng.choice(u, (10000, len(u))).mean(1), [.025, .975])
            elif k == "P1":
                s1 = g[g.seed == g.seed.min()]; r["ci_lo"], r["ci_hi"] = wilson_ci(int(s1.ok.sum()), len(s1))
            else:
                u = g[g.seed == g.seed.min()].groupby("fold").ok.mean(); r["fold_sd"] = u.std(ddof=1)
                r["ci_lo"], r["ci_hi"] = np.quantile(rng.choice(u.values, (10000, len(u))).mean(1), [.025, .975])
            rows.append(r)
    T5 = pd.DataFrame(rows); T5.to_csv(f"{TABLES}/table5_main.csv", index=False)
    pw = []
    def pair(k, a, b):
        p = P[k]; pa, pb = p[p.model == a], p[p.model == b]
        if pa.empty or pb.empty: return
        if k == "P1":
            ps = [mcnemar_exact(pa[pa.seed == s].sort_values("sample_id").ok.values, pb[pb.seed == s].sort_values("sample_id").ok.values)["p_value"] for s in sorted(set(pa.seed) & set(pb.seed))]
            pw.append(dict(protocol=k, A=a, B=b, test="exact McNemar per seed", mean_diff=pa.groupby("seed").ok.mean().mean() - pb.groupby("seed").ok.mean().mean(), p_min=min(ps), p_max=max(ps)))
        elif k == "P2":
            s = min(set(pa.seed) & set(pb.seed)); fa, fb = pa[pa.seed == s].groupby("fold").ok.mean(), pb[pb.seed == s].groupby("fold").ok.mean()
            common = fa.index.intersection(fb.index)
            if len(common) < 2: return
            r = nadeau_bengio_ttest(fa.loc[common].values, fb.loc[common].values, 360, 100)
            pw.append(dict(protocol=k, A=a, B=b, test="Nadeau-Bengio t (10 folds)", mean_diff=r["mean_diff"], p_min=r["p_value"], p_max=r["p_value"]))
        else:
            ua, ub = pa.groupby(["grade", "batch_id"]).ok.mean(), pb.groupby(["grade", "batch_id"]).ok.mean(); d = (ua - ub.loc[ua.index]).values
            pv = sps.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
            pw.append(dict(protocol=k, A=a, B=b, test="Wilcoxon over 25 solutions", mean_diff=d.mean(), p_min=pv, p_max=pv))
    for k in P:
        for b in ["cnn_flatten", "cnn_lstm", "cnn_gru", "cnn_gap", "SVM-RBF", "RF"]: pair(k, "cnn_bigru", b)
        pair(k, "SVM-RBF", "cnn_gap")
    PW = pd.DataFrame(pw); PW.to_csv(f"{TABLES}/pairwise_tests.csv", index=False)
    if ex("orig_global_legacy") and ex("orig_train_only"):
        a, b = load("orig_global_legacy"), load("orig_train_only")
        se = pd.DataFrame({"legacy_global": a.groupby(["model", "seed"]).ok.mean().groupby("model").mean(), "train_only": b.groupby(["model", "seed"]).ok.mean().groupby("model").mean()})
        se["diff"] = se.legacy_global - se.train_only; se.to_csv(f"{TABLES}/scaler_effect.csv"); print("\nscaler effect (P1):\n", se.round(3))
    rec = P["P3"].groupby(["model", "grade"]).ok.mean().unstack(); rec.to_csv(f"{TABLES}/grouped_recall_by_grade.csv")
    pd.set_option("display.width", 220)
    print("\nTABLE 5\n", T5.round(3).to_string(), "\n\nPAIRWISE\n", PW.round(4).to_string(), "\n\nRECALL BY GRADE (P3)\n", rec.round(2))
    return T5
if RUN["stats"]:
    T5 = compute_stats()
    for d in ("orig_train_only", "grouped"):                       # leakage report headline
        f = f"{RUNS}/{d}/leakage_report.json"
        if os.path.exists(f):
            lr = json.load(open(f)); print(f"\n{d}: test recordings whose solution is also in training: {sum(x['test_samples_whose_batch_is_also_in_train'] for x in lr)} of {sum(x['n_test'] for x in lr)}")


scaler effect (P1):
              legacy_global  train_only   diff
model                                        
cnn_bigru            0.974       0.952  0.021
cnn_flatten          0.820       0.926 -0.106
cnn_gap              0.974       0.979 -0.005
cnn_gru              0.974       0.958  0.016
cnn_lstm             0.942       0.952 -0.011

TABLE 5
    protocol        model  n_seeds  acc_mean  acc_sd_seeds  ci_lo  ci_hi  fold_sd
0        P1  cnn_flatten        3     0.926         0.040  0.788  0.945      NaN
1        P1     cnn_lstm        3     0.952         0.016  0.869  0.984      NaN
2        P1      cnn_gru        3     0.958         0.009  0.891  0.991      NaN
3        P1    cnn_bigru        3     0.952         0.032  0.827  0.966      NaN
4        P1      cnn_gap        3     0.979         0.009  0.891  0.991      NaN
5        P1      SVM-RBF        3     0.984         0.000  0.915  0.997      NaN
6        P1           RF        3     0.974         0.009  0.915  0.997      Na

In [11]:
# %% 9. Figures: accuracy by protocol, and the confusion matrix under solution-held-out testing
if RUN["figures"] and os.path.exists(f"{TABLES}/table5_main.csv"):
    import matplotlib; matplotlib.use("Agg");
    import matplotlib.pyplot as plt
    T = pd.read_csv(f"{TABLES}/table5_main.csv");
    mods = [m for m in ["cnn_flatten", "cnn_lstm", "cnn_gru", "cnn_bigru", "cnn_gap", "SVM-RBF", "RF"] if m in set(T.model)]
    fig, ax = plt.subplots(figsize=(8, 3.6));
    w = 0.26;
    cols = {"P1": "#0072B2", "P2": "#E69F00", "P3": "#D55E00"}
    for i, k in enumerate(["P1", "P2", "P3"]):
        v = [T[(T.protocol == k) & (T.model == m)].acc_mean.mean() * 100 for m in mods]
        ax.bar(np.arange(len(mods)) + (i - 1) * w, v, w, label={"P1": "P1 single split (recordings)", "P2": "P2 repeated 5-fold (recordings)", "P3": "P3 solution-held-out"}[k], color=cols[k])
    ax.axhline(20, color="grey", ls=":", lw=1);
    ax.set_xticks(range(len(mods)));
    ax.set_xticklabels(mods, rotation=20);
    ax.set_ylabel("Accuracy (%)");
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout();
    plt.savefig(f"{FIGS}/fig_accuracy_by_protocol.png", dpi=300);
    plt.close()
    p3 = f"{RUNS}/grouped/confusion_cnn_bigru.csv"
    if os.path.exists(p3):
        cm = pd.read_csv(p3, index_col=0).to_numpy(); fig, ax = plt.subplots(figsize=(3.6, 3.2)); ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
        for i in range(5):
            for j in range(5): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
        ax.set_xticks(range(5));
        ax.set_xticklabels(range(1, 6));
        ax.set_yticks(range(5));
        ax.set_yticklabels(range(1, 6));
        ax.set_xlabel("Predicted grade");
        ax.set_ylabel("True grade");
        ax.set_title("CNN-BiGRU, solution-held-out")
        plt.tight_layout(); plt.savefig(f"{FIGS}/fig_confusion_grouped.png", dpi=300);
        plt.close()
    print("figures saved to", FIGS)

figures saved to /content/enose_repro/results/figures


In [12]:
# %% 9b. Manuscript figures: Rs/R0 signals (Fig. 2), within-batch drift (Fig. 5), confusion matrices (Fig. 6), training curves (Section 4.9), framework and architectures (Figs. 3-4)
# Needs only the array, the prediction files and the history_*.csv files that run_protocol() wrote. Each figure is skipped if its input is missing.
if RUN.get("manuscript_figs", True):
    import glob, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    MF = f"{FIGS}/manuscript"; os.makedirs(MF, exist_ok=True)
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
                         "xtick.labelsize": 10, "ytick.labelsize": 10, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 10})
    GC = ["#1b6ca8", "#e08a1e", "#2a9d6f", "#c0392b", "#7b5ea7"]
    NICE = {"cnn_flatten": "CNN-Flatten", "cnn_lstm": "CNN-LSTM", "cnn_gru": "CNN-GRU", "cnn_bigru": "CNN-BiGRU", "cnn_gap": "CNN-GAP"}
    Xf, yf, metaf = load_dataset(NPZ)
    # --- Fig. 2: mean +- SD of Rs/R0 over the 700 s, per sensor and grade (100 recordings per grade)
    ch = ["MQ2", "MQ3", "MQ4", "MQ7", "MQ8", "MQ135", "MQ137", "TGS2602"]; t = np.arange(Xf.shape[1])
    fig, axs = plt.subplots(4, 2, figsize=(3.15, 5.6), sharex=True)
    for c, a in enumerate(axs.T.flat):
        for g in range(5):
            m = Xf[yf == g, :, c]; mu, sd = m.mean(0), m.std(0)
            a.plot(t, mu, color=GC[g], lw=1, label=f"Grade {g + 1}");
            a.fill_between(t, mu - sd, mu + sd, color=GC[g], alpha=.1, lw=0)
        a.set_title(ch[c], loc="left", pad=1);
        a.locator_params(axis="y", nbins=3);
        a.set_xticks([0, Xf.shape[1] // 2, Xf.shape[1]])

    for a in axs[-1]:
      a.set_xlabel("Time (s)")
    for a in axs[:, 0]:
      a.set_ylabel("Rs/R0")

    h_, l_ = axs[0, 0].get_legend_handles_labels();
    fig.legend(h_, l_, ncol=3, frameon=False, loc="lower center", handlelength=1.0, columnspacing=.8, borderaxespad=0)
    plt.tight_layout(pad=.3, h_pad=.4, w_pad=.6, rect=(0, .09, 1, 1)); plt.savefig(f"{MF}/fig2_raw_signals.png", dpi=300);
    plt.close()
    # --- Fig. 5: mean MQ135 response (600-700 s) against cycle number, one line per batch (solution), Grade 1 and Grade 3
    fig, axs = plt.subplots(2, 1, figsize=(3.15, 3.9), sharex=True)
    for a, gr in zip(axs, [0, 2]):
        for bt in range(5):
            i = (yf == gr) & (metaf.batch_id.values == bt); o = np.argsort(metaf.run_order.values[i])
            a.plot(metaf.run_order.values[i][o] + 1, Xf[i][o][:, 600:700, 5].mean(1), lw=1, marker="o", ms=3, label=f"B{bt + 1}")
        a.set_title(f"Grade {gr + 1}", loc="left", pad=1); a.set_ylabel("Rs/R0")
    axs[1].set_xlabel("Cycle within batch"); axs[0].legend(ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(.5, 1.45), handlelength=.8, columnspacing=.6, handletextpad=.3)
    plt.tight_layout(pad=.3, h_pad=.5); plt.savefig(f"{MF}/fig5_within_batch_drift.png", dpi=300); plt.close()
    # --- Fig. 6: row-normalised confusion matrices (percent), pooled over seeds, P1 (top) and P3 (bottom)
    def cmat(d, m):
        f = f"{RUNS}/{d}/predictions.csv"
        if not os.path.exists(f):
          return None
        p = pd.read_csv(f);
        g = p[p.model == m];
        c = confusion_matrix(g.y_true, g.y_pred, labels=range(5)).astype(float);
        return c / c.sum(1, keepdims=True)

    if cmat("orig_train_only", "cnn_bigru") is not None and cmat("grouped", "cnn_bigru") is not None:
        fig, axs = plt.subplots(2, 3, figsize=(6.8, 5.4))
        for r, (d, lab) in enumerate([("orig_train_only", "P1"), ("grouped", "P3")]):
            for a, m in zip(axs[r], ["cnn_flatten", "cnn_bigru", "cnn_gap"]):
                c = cmat(d, m); a.imshow(c, vmin=0, vmax=1, cmap="Blues")
                for i in range(5):
                    for j in range(5): a.text(j, i, f"{round(100 * c[i, j])}", ha="center", va="center", fontsize=10, color="w" if c[i, j] > .5 else "k")
                a.set_title(f"{NICE[m]}, {lab}"); a.set_xticks(range(5)); a.set_yticks(range(5)); a.set_xticklabels(range(1, 6)); a.set_yticklabels(range(1, 6))
                for s in a.spines.values(): s.set_visible(False)
                if r == 1: a.set_xlabel("Predicted grade")
                if m == "cnn_flatten": a.set_ylabel("True grade")
        plt.tight_layout(pad=.4, h_pad=1.4, w_pad=1.2); plt.savefig(f"{MF}/fig6_confusion.png", dpi=300); plt.close()

    # --- Training curves (the manuscript, Section 4.9, refers to these): P1 seed 1 for every model; loss and accuracy, training vs validation
    for d, tag in [("orig_train_only", "P1"), ("grouped", "P3")]:
        hs = {m: sorted(glob.glob(f"{RUNS}/{d}/history_{m}_*_s1.csv")) for m in ALL_MODELS}
        if not any(hs.values()): continue
        fig, ax = plt.subplots(2, len(ALL_MODELS), figsize=(11, 4.6), sharex=True)
        for j, m in enumerate(ALL_MODELS):
            for k, f in enumerate(hs[m]):
                h = pd.read_csv(f)
                ax[0, j].plot(h.epoch, h.loss, lw=1, color="#1b6ca8", alpha=.9 if tag == "P1" else .5, label="training" if k == 0 else None)
                ax[0, j].plot(h.epoch, h.val_loss, lw=1, color="#c0392b", alpha=.9 if tag == "P1" else .5, label="validation" if k == 0 else None)
                ax[1, j].plot(h.epoch, h.accuracy, lw=1, color="#1b6ca8", alpha=.9 if tag == "P1" else .5); ax[1, j].plot(h.epoch, h.val_accuracy, lw=1, color="#c0392b", alpha=.9 if tag == "P1" else .5)
            ax[0, j].set_title(NICE[m]); ax[1, j].set_xlabel("Epoch")
        ax[0, 0].set_ylabel("Loss"); ax[1, 0].set_ylabel("Accuracy"); ax[0, 0].legend(frameon=False)
        fig.suptitle(f"{tag}: training and validation curves (seed 1" + ("" if tag == "P1" else "; one line per fold") + ")", fontsize=10)
        plt.tight_layout(); plt.savefig(f"{MF}/fig_training_curves_{tag}.png", dpi=300); plt.close()

    # --- Fig. 3 (framework) and Fig. 4 (architectures) as drawings
    def bx(ax, x, y_, w, h, txt, fc, fs=10, **k):
      ax.add_patch(FancyBboxPatch((x, y_), w, h, boxstyle="round,pad=0.01,rounding_size=0.12", fc=fc, ec="#2c3e50", lw=.8));
      ax.text(x + w / 2, y_ + h / 2, txt, ha="center", va="center", fontsize=fs, **k)

    def ar(ax, x0, y0, x1, y1):
      ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=.9, color="#2c3e50", shrinkA=0, shrinkB=0))

    fig, ax = plt.subplots(figsize=(3.15, 4.6));
    ax.set_xlim(0, 10);
    ax.set_ylim(0, 14);
    ax.axis("off")

    bx(ax, .2, 12.4, 9.6, 1.5, "Raw response, 700 × 8", "#fdebd0");
    ar(ax, 5, 12.4, 5, 11.6);
    bx(ax, .2, 10.1, 9.6, 1.5, "Training-only z-score", "#d6eaf8")
    ar(ax, 2.4, 10.1, 2.4, 9.2);
    ar(ax, 7.6, 10.1, 7.6, 9.2)
    bx(ax, .2, 6.4, 4.6, 2.8, "1D-CNN and\ncore layer\n(5 models)", "#d5f5e3");
    bx(ax, 5.2, 6.4, 4.6, 2.8, "80 features,\nSVM, RF,\nLogReg, MLP", "#fef5e7");
    ar(ax, 5, 6.4, 5, 5.5)
    ax.text(5, 5., "Same test indices", ha="center", va="center", fontsize=10, style="italic")
    bx(ax, .2, .2, 3., 3.8, "P1\nsingle\nsplit", "#eaf2f8");
    bx(ax, 3.5, .2, 3., 3.8, "P2\n5-fold\n× 2", "#eaf2f8");
    bx(ax, 6.8, .2, 3., 3.8, "P3\nbatch-\ngrouped", "#fadbd8")
    plt.savefig(f"{MF}/fig3_framework.png", dpi=300); plt.close()
    fig, ax = plt.subplots(figsize=(6.8, 4.3));
    ax.set_xlim(0, 20);
    ax.set_ylim(-1, 12.5); ax.axis("off")
    bx(ax, 4, 11., 12, 1.4, "Input (700, 8)", "#fdebd0");
    bx(ax, 2, 8.3, 16, 2.1, "Conv1D(32, k=7), BatchNorm, MaxPool(2)\nConv1D(64, k=5), BatchNorm, MaxPool(2) → (175, 64)", "#d6eaf8"); ar(ax, 10, 11., 10, 10.45)
    for k, (n, c, p) in enumerate([("CNN-\nFlatten", "Flatten\n(11,200)\nDrop 0.2", "729,509"), ("CNN-\nLSTM", "LSTM(64)\nDrop 0.4", "49,829"), ("CNN-\nGRU", "GRU(64)\nDrop 0.4", "41,765"),
                                   ("CNN-\nBiGRU", "BiGRU\n(64+64)\nDrop 0.4", "70,821"), ("CNN-\nGAP", "GAP\n(64)\nDrop 0.4", "16,805")]):
        x = .15 + k * 3.98;
        ax.annotate("", xy=(x + 1.85, 7.1), xytext=(min(max(x + 1.85, 3), 17), 8.3), arrowprops=dict(arrowstyle="-|>", lw=.8, color="#2c3e50", shrinkA=0, shrinkB=0))
        bx(ax, x, 3.9, 3.7, 3.2, c, "#d5f5e3");
        ax.text(x + 1.85, 2.9, n, ha="center", va="center", fontsize=10, fontweight="bold");
        ax.text(x + 1.85, 1.75, p, ha="center", va="center", fontsize=10, color="#8e1b4a", style="italic")
    ax.text(10, 0., "Head: Dense(64, ReLU), Dropout (0.2 Flatten, 0.3 others), Dense(5, Softmax)\nItalics: trainable parameters", ha="center", va="center", fontsize=10)
    plt.savefig(f"{MF}/fig4_architectures.png", dpi=300);
    plt.close()
    print("manuscript figures:", sorted(os.listdir(MF)))

manuscript figures: ['fig2_raw_signals.png', 'fig3_framework.png', 'fig4_architectures.png', 'fig5_within_batch_drift.png', 'fig6_confusion.png', 'fig_training_curves_P1.png', 'fig_training_curves_P3.png']


In [13]:
# %% 10. Optional: CPU inference latency (run on an idle CPU)
if RUN["latency"]:
    out = []
    for m in ALL_MODELS:
        model = build_model(m); tr, nt, tot = count_params(model); x1 = np.random.randn(1, 700, 8).astype("float32"); x32 = np.random.randn(32, 700, 8).astype("float32")
        f = tf.function(lambda x: model(x, training=False)); f(x1); f(x32); t1, t32 = [], []
        for _ in range(200): s = time.perf_counter(); f(x1).numpy(); t1.append((time.perf_counter() - s) * 1e3)
        for _ in range(50): s = time.perf_counter(); f(x32).numpy(); t32.append((time.perf_counter() - s) * 1e3)
        out.append(dict(model=m, trainable=tr, weights_MB_fp32=round(tot * 4 / 2 ** 20, 3), latency_ms_b1_median=round(float(np.median(t1)), 2), throughput_per_s_b32=round(32 / (np.median(t32) / 1e3), 1)))
    pd.DataFrame(out).to_csv(f"{TABLES}/inference_latency.csv", index=False); print(pd.DataFrame(out))

In [ ]:
# %% 11. Revision controls E1-E4 (code of extra_exp.py is embedded below and written to WORK/code)
# Needs P1 (orig_train_only) to be finished first: E1 takes the fixed epoch counts from its early-stopping epochs.
import subprocess
CODE = f"{WORK}/code"; os.makedirs(CODE, exist_ok=True); ROOT = f"{WORK}/results"          # ROOT contains runs/ (main experiments)
os.makedirs(f"{ROOT}/full", exist_ok=True)
if not os.path.exists(f"{ROOT}/full/real.npz"): os.symlink(NPZ, f"{ROOT}/full/real.npz")
SRC = {}
SRC["extra_exp.py"] = r'''"""Additional controls for the IJIES revision (reviewer 1 and 2).

Experiments (each is resumable; results are appended row by row to OUT/<exp>/predictions.csv):
  e1_matched   : P1 with matched training data.  Deep models are retrained on all 437 train+val recordings for a fixed
                 number of epochs (mean best epoch of the 375-recording runs); handcrafted classifiers are refit on the
                 375 training recordings only (the same recordings the deep models update their weights on).
  e2_random    : recording-level random control with the same budget as P3: 300 train / 100 validation / 100 test,
                 stratified by grade, same train-only standardiser, optimiser and early stopping (10 random splits).
  e3_<variant> : capacity- and regularisation-matched controls, evaluated under P1 (375/62/63) and P3.
  e4_<model>   : contemporary non-recurrent baselines (TCN, small Transformer encoder) under P1, P2 and P3, and E2.
"""
import os, sys, json, argparse, random, time
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

NPZ = os.environ.get("ENOSE_NPZ", f"{os.getcwd()}/full/real.npz")
OUT = os.environ.get("ENOSE_OUT", f"{os.getcwd()}/runs_extra")
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 32, 100, 12
tf.config.threading.set_intra_op_parallelism_threads(int(os.environ.get("TF_THREADS", 1)))
tf.config.threading.set_inter_op_parallelism_threads(1)

# ---------------------------------------------------------------- data
def load_dataset(path=NPZ):
    d = np.load(path); X, y = d["X"].astype(np.float32), d["y"].astype(int)
    meta = pd.DataFrame({"sample_id": np.arange(len(y)), "grade": y, "batch_id": d["batch_id"].astype(int), "run_order": d["run_order"].astype(int)})
    meta["group"] = meta["grade"] * 100 + meta["batch_id"]
    return X, y, meta
def fit_standardizer(Xt):
    flat = Xt.reshape(-1, Xt.shape[-1]); mu, sd = flat.mean(0), flat.std(0); sd[sd == 0] = 1.0; return mu.astype(np.float32), sd.astype(np.float32)
def apply_standardizer(X, mu, sd): return ((X - mu) / sd).astype(np.float32)

# ---------------------------------------------------------------- splits
def original_split(y, seed=42):
    yc = np.eye(int(y.max()) + 1)[y]; idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.25, random_state=seed, stratify=yc)
    va, te = train_test_split(tmp, test_size=0.5, random_state=seed, stratify=yc[tmp])
    return [dict(name="original_seed42", train=tr, val=va, test=te)]
def rkf_splits(y, n_repeats=2, seed=0, val_fraction=0.1):
    out = []
    for i, (a, te) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=seed).split(np.zeros(len(y)), y)):
        tr, va = train_test_split(a, test_size=val_fraction, stratify=y[a], random_state=seed + i)
        out.append(dict(name=f"rskf_r{i // 5}_f{i % 5}", train=tr, val=va, test=te))
    return out
def grouped_splits(meta, n=5):
    b = meta["batch_id"].to_numpy()
    return [dict(name=f"grouped_test_batch{k}", train=np.where((b != k) & (b != (k + 1) % n))[0], val=np.where(b == (k + 1) % n)[0], test=np.where(b == k)[0]) for k in range(n)]
def random_budget_splits(y, n_splits=10, n_train=300, n_val=100, n_test=100, seed=1000):
    """Recording-level random splits with exactly the P3 budget (300/100/100), stratified by grade (60/20/20 per grade)."""
    out = []; idx = np.arange(len(y))
    for i in range(n_splits):
        a, te = train_test_split(idx, test_size=n_test, stratify=y, random_state=seed + i)
        tr, va = train_test_split(a, test_size=n_val, stratify=y[a], random_state=seed + i)
        out.append(dict(name=f"random300_s{i}", train=tr, val=va, test=te))
    return out

# ---------------------------------------------------------------- models
def _conv(x, f, k):
    x = layers.Conv1D(f, k, padding="same", activation="relu")(x); x = layers.BatchNormalization()(x); return layers.MaxPooling1D(2)(x)
BASE_DROP = {"cnn_flatten": (0.2, 0.0, 0.2), "cnn_lstm": (0.0, 0.4, 0.3), "cnn_bigru": (0.0, 0.4, 0.3), "cnn_gru": (0.0, 0.4, 0.3), "cnn_gap": (0.0, 0.4, 0.3)}
def _frontend_head(name, core, d, dense=64, input_shape=(700, 8)):
    inp = keras.Input(shape=input_shape); x = _conv(inp, 32, 7); x = _conv(x, 64, 5)
    if d[0]: x = layers.Dropout(d[0])(x)
    x = core(x)
    if d[1]: x = layers.Dropout(d[1])(x)
    x = layers.Dense(dense, activation="relu")(x); x = layers.Dropout(d[2])(x)
    return keras.Model(inp, layers.Dense(5, activation="softmax")(x), name=name)
def tcn_block(x, f, k, dil, drop):
    r = x
    for _ in range(2):
        x = layers.Conv1D(f, k, padding="causal", dilation_rate=dil)(x); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x); x = layers.Dropout(drop)(x)
    if r.shape[-1] != f: r = layers.Conv1D(f, 1)(r)
    return layers.Activation("relu")(layers.Add()([x, r]))
def build_tcn(input_shape=(700, 8)):
    inp = keras.Input(shape=input_shape); x = layers.MaxPooling1D(4)(inp)          # 175 steps: same temporal budget as the CNN front end
    for f, dil in [(32, 1), (48, 2), (64, 4), (64, 8)]: x = tcn_block(x, f, 5, dil, 0.2)
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x); x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(5, activation="softmax")(x), name="tcn")
def build_transformer(input_shape=(700, 8), d_model=64, heads=4, ff=128, n_layers=2):
    inp = keras.Input(shape=input_shape); x = _conv(inp, 32, 7); x = _conv(x, d_model, 5)          # 175 tokens x 64
    pos = layers.Embedding(175, d_model)(tf.range(175)); x = x + pos
    for _ in range(n_layers):
        h = layers.LayerNormalization(epsilon=1e-6)(x); h = layers.MultiHeadAttention(heads, d_model // heads, dropout=0.1)(h, h); x = x + layers.Dropout(0.1)(h)
        h = layers.LayerNormalization(epsilon=1e-6)(x); h = layers.Dense(ff, activation="relu")(h); h = layers.Dropout(0.1)(h); h = layers.Dense(d_model)(h); x = x + h
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x); x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(5, activation="softmax")(x), name="transformer")
def n_trainable(m): return int(sum(w.numpy().size for w in m.trainable_weights))
def build_model(name, input_shape=(700, 8)):
    if name in BASE_DROP:
        d = BASE_DROP[name]
        core = {"cnn_flatten": lambda x: layers.Flatten()(x), "cnn_lstm": lambda x: layers.LSTM(64)(x), "cnn_bigru": lambda x: layers.Bidirectional(layers.GRU(64))(x),
                "cnn_gru": lambda x: layers.GRU(64)(x), "cnn_gap": lambda x: layers.GlobalAveragePooling1D()(x)}[name]
        return _frontend_head(name, core, d, 64, input_shape)
    if name == "flatten_reg04":   # CNN-Flatten with the regularisation of the recurrent models (core dropout 0.4, dense dropout 0.3)
        return _frontend_head(name, lambda x: layers.Flatten()(x), (0.0, 0.4, 0.3), 64, input_shape)
    if name == "flatten_pool8":   # CNN-Flatten with an extra pooling so that the head has roughly the capacity of CNN-BiGRU
        return _frontend_head(name, lambda x: layers.Flatten()(layers.MaxPooling1D(8)(x)), (0.0, 0.4, 0.3), 64, input_shape)
    if name == "gap_wide":        # CNN-GAP with a wider dense layer: trainable parameters close to CNN-BiGRU
        return _frontend_head(name, lambda x: layers.GlobalAveragePooling1D()(x), (0.0, 0.4, 0.3), 800, input_shape)
    if name == "bigru_u32":       # CNN-BiGRU with fewer units: closer to CNN-GRU / CNN-LSTM capacity
        return _frontend_head(name, lambda x: layers.Bidirectional(layers.GRU(32))(x), (0.0, 0.4, 0.3), 64, input_shape)
    if name == "gru_u96":         # CNN-GRU with more units: trainable parameters close to CNN-BiGRU
        return _frontend_head(name, lambda x: layers.GRU(96)(x), (0.0, 0.4, 0.3), 64, input_shape)
    if name == "tcn": return build_tcn(input_shape)
    if name == "transformer": return build_transformer(input_shape)
    raise ValueError(name)

# ---------------------------------------------------------------- training
def set_seed(s): random.seed(s); np.random.seed(s); tf.random.set_seed(s)
def train_one(model_name, seed, Xtr, ytr, Xva, yva, fixed_epochs=None):
    set_seed(seed); m = build_model(model_name, Xtr.shape[1:])
    m.compile(optimizer=keras.optimizers.Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
    if fixed_epochs:
        m.fit(Xtr, keras.utils.to_categorical(ytr, 5), epochs=fixed_epochs, batch_size=BATCH, verbose=0); ep = fixed_epochs
    else:
        h = m.fit(Xtr, keras.utils.to_categorical(ytr, 5), validation_data=(Xva, keras.utils.to_categorical(yva, 5)), epochs=MAX_EPOCHS, batch_size=BATCH, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)])
        ep = len(h.history["loss"])
    return m, ep

def hc_features(X):
    t = np.arange(X.shape[1]); trap = getattr(np, "trapezoid", None) or np.trapz
    sl = ((t - t.mean())[None, :, None] * (X - X.mean(1, keepdims=True))).sum(1) / ((t - t.mean()) ** 2).sum()
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1), X.argmax(1) / X.shape[1], sl, trap(X, axis=1) / X.shape[1],
                           X[:, :100].mean(1), X[:, -100:].mean(1), np.abs(np.diff(X, axis=1)).max(1)], 1)
HC = {"RF": lambda s: RandomForestClassifier(300, random_state=s, n_jobs=1),
      "SVM-RBF": lambda s: make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", random_state=s)),
      "LogReg": lambda s: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=s)),
      "MLP": lambda s: make_pipeline(StandardScaler(), MLPClassifier((64,), max_iter=500, early_stopping=True, random_state=s))}

class Store:
    def __init__(self, exp):
        self.dir = f"{OUT}/{exp}"; os.makedirs(self.dir, exist_ok=True); self.path = f"{self.dir}/predictions.csv"
        self.done = set()
        if os.path.exists(self.path):
            p = pd.read_csv(self.path, usecols=["model", "seed", "fold"]).drop_duplicates(); self.done = set(map(tuple, p.values.tolist()))
    def has(self, model, seed, fold): return (model, seed, fold) in self.done
    def add(self, rows, extra=None):
        df = pd.DataFrame(rows); df.to_csv(self.path, mode="a", header=not os.path.exists(self.path), index=False)
        self.done.add((rows[0]["model"], rows[0]["seed"], rows[0]["fold"]))
        if extra:
            with open(f"{self.dir}/epochs.csv", "a") as f: f.write(",".join(map(str, extra)) + "\n")

def deep_run(exp, splits, models, seeds, scaler_on="train", fixed_epochs=None, train_plus_val=False):
    X, y, meta = load_dataset(); st = Store(exp)
    for sp in splits:
        tr = np.concatenate([sp["train"], sp["val"]]) if train_plus_val else sp["train"]
        mu, sd = fit_standardizer(X[tr]); Xtr, Xva, Xte = (apply_standardizer(X[i], mu, sd) for i in (tr, sp["val"], sp["test"]))
        for mname in models:
            for s in seeds:
                if st.has(mname, s, sp["name"]): continue
                t0 = time.time(); fe = fixed_epochs[mname] if isinstance(fixed_epochs, dict) else fixed_epochs
                m, ep = train_one(mname, s, Xtr, y[tr], Xva, y[sp["val"]], fe); prob = m.predict(Xte, verbose=0); pred = prob.argmax(1)
                rows = []
                for j, i in enumerate(sp["test"]):
                    r = dict(model=mname, seed=s, fold=sp["name"], sample_id=int(i), grade=int(y[i]), batch_id=int(meta.batch_id[i]), y_true=int(y[i]), y_pred=int(pred[j]))
                    r.update({f"p_grade{c + 1}": float(prob[j, c]) for c in range(5)}); rows.append(r)
                st.add(rows, extra=[mname, s, sp["name"], ep, n_trainable(m), f"{time.time() - t0:.1f}"])
                print(f"[{exp}] {sp['name']:<22} {mname:<14} s={s} acc={(pred == y[sp['test']]).mean():.3f} ep={ep} ({time.time() - t0:.0f}s)", flush=True)
                keras.backend.clear_session()

def hc_run(exp, splits, seeds=(1, 2, 3), use_val=False):
    X, y, meta = load_dataset(); F = hc_features(X); st = Store(exp)
    for sp in splits:
        tr = np.concatenate([sp["train"], sp["val"]]) if use_val else sp["train"]
        for mname, mk in HC.items():
            for s in seeds:
                if st.has(mname, s, sp["name"]): continue
                pred = mk(s).fit(F[tr], y[tr]).predict(F[sp["test"]]); rows = []
                for j, i in enumerate(sp["test"]):
                    rows.append(dict(model=mname, seed=s, fold=sp["name"], sample_id=int(i), grade=int(y[i]), batch_id=int(meta.batch_id[i]), y_true=int(y[i]), y_pred=int(pred[j])))
                st.add(rows); print(f"[{exp}] {sp['name']:<22} {mname:<8} s={s} acc={(pred == y[sp['test']]).mean():.3f}", flush=True)

def best_epochs():
    m = pd.read_csv(f"{os.getcwd()}/runs/orig_train_only/metrics_per_fold.csv")
    return {k: int(round(v - PATIENCE)) for k, v in m.groupby("model").epochs_run.mean().items()}

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("exp"); a = ap.parse_args()
    X, y, meta = load_dataset(); DEEP5 = ["cnn_flatten", "cnn_lstm", "cnn_gru", "cnn_bigru", "cnn_gap"]; S3 = [1, 2, 3]
    if a.exp == "params":
        print({n: n_trainable(build_model(n)) for n in DEEP5 + ["flatten_reg04", "flatten_pool8", "gap_wide", "bigru_u32", "gru_u96", "tcn", "transformer"]})
    elif a.exp == "e1_deep":     deep_run("e1_deep437", original_split(y), DEEP5, S3, fixed_epochs=best_epochs(), train_plus_val=True)
    elif a.exp == "e1_hc":       hc_run("e1_hc375", original_split(y), use_val=False)
    elif a.exp == "e2_deep":     deep_run("e2_random_deep", random_budget_splits(y), DEEP5, [1])
    elif a.exp == "e2_hc":       hc_run("e2_random_hc", random_budget_splits(y), seeds=(1,), use_val=False)
    elif a.exp == "e3_p1":       deep_run("e3_p1", original_split(y), ["flatten_reg04", "flatten_pool8", "gap_wide", "bigru_u32", "gru_u96"], S3)
    elif a.exp == "e3_p3":       deep_run("e3_p3", grouped_splits(meta), ["flatten_reg04", "flatten_pool8", "gap_wide", "bigru_u32", "gru_u96"], S3)
    elif a.exp == "e3_e2":       deep_run("e3_random", random_budget_splits(y), ["flatten_reg04", "flatten_pool8", "gap_wide", "bigru_u32", "gru_u96"], [1])
    elif a.exp == "e4_p1":       deep_run("e4_p1", original_split(y), ["tcn", "transformer"], S3)
    elif a.exp == "e4_p3":       deep_run("e4_p3", grouped_splits(meta), ["tcn", "transformer"], S3)
    elif a.exp == "e4_p2":       deep_run("e4_p2", rkf_splits(y, 2), ["tcn", "transformer"], [1])
    elif a.exp == "e4_e2":       deep_run("e4_random", random_budget_splits(y), ["tcn", "transformer"], [1])
    else: raise SystemExit("unknown exp")
'''
SRC["stats_v2.py"] = r'''"""Revised statistical analysis for the IJIES revision.  Reads per-recording predictions, writes tables to stats_v2/.

Definitions (written to stats_v2/METHODS.md):
 * unit of analysis for P3: the 25 test batches (5 grades x 5 batches).  A batch accuracy is the accuracy over its 20 recordings,
   averaged over the three training seeds BEFORE any resampling or test, so seeds are never treated as replicates.
 * cluster bootstrap: 10,000 draws, rng seed 0, percentile 95% interval, paired contrasts reuse the same resampled indices for both models.
   Three schemes are computed: (i) PLAIN (primary, as in the first version): 25 batches resampled with replacement; (ii) grade-stratified: the 5 batches
   within each grade are resampled (design-based, narrower); (iii) fold-cluster: whole folds (5 batches tested by the same fitted model) are resampled
   (conservative, only 5 clusters).
 * dependence: batches tested by the same fitted model (same fold) and training sets that overlap between folds are not independent;
   all P3 intervals/p-values are therefore exploratory.  A fold-level analysis (5 folds, sign-flip permutation, minimum two-sided p = 0.0625)
   is reported next to the batch-level one as a more conservative bound.
 * P1: per-seed exact McNemar; the P1 p-value used in Holm is the LARGEST of the three seed p-values.
 * P2: Nadeau-Bengio corrected resampled t-test on ten fold accuracies (n_test/n_train = 100/360).
 * Holm: applied within each protocol to the family of 7 primary contrasts listed in FAMILY.
 * Practical margin: +-5 percentage points (chosen as the smallest difference we would regard as architecturally relevant: 5 pp is
   one recording in 20 per batch, 3 recordings in 63 for P1); 90% interval within the margin = equivalence supported.
"""
import os, json, itertools
import numpy as np, pandas as pd
from scipy import stats as sps
from sklearn.metrics import precision_recall_fscore_support

SP = os.environ.get("ENOSE_ROOT", os.getcwd())  # working root: contains runs/ (main experiments) and full/real.npz
R, RX, OUT = f"{SP}/runs", f"{SP}/runs_extra", f"{SP}/extra/stats_v2"; os.makedirs(OUT, exist_ok=True)
DEEP5 = ["cnn_flatten", "cnn_lstm", "cnn_gru", "cnn_bigru", "cnn_gap"]; HC4 = ["SVM-RBF", "RF", "LogReg", "MLP"]
FAMILY = [("cnn_bigru", "cnn_flatten"), ("cnn_bigru", "cnn_lstm"), ("cnn_bigru", "cnn_gru"), ("cnn_bigru", "cnn_gap"),
          ("SVM-RBF", "cnn_bigru"), ("RF", "cnn_bigru"), ("SVM-RBF", "cnn_gap")]
MARGIN = 0.05; NBOOT = 10000

def load(*dirs, base=R):
    ps = [pd.read_csv(f"{base}/{d}/predictions.csv") for d in dirs if os.path.exists(f"{base}/{d}/predictions.csv")]
    if not ps: return pd.DataFrame()
    p = pd.concat(ps, ignore_index=True); p["ok"] = (p.y_true == p.y_pred).astype(int); return p
P1 = load("orig_train_only", "base_original"); P2 = load("rkf", "rkf2", "base_rkf"); P3 = load("grouped", "base_batch_grouped")

def holm(pvals):
    p = np.asarray(pvals, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(1, run)
    return adj
def mcnemar_exact(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool); n01, n10 = int((a & ~b).sum()), int((~a & b).sum()); n = n01 + n10
    return 1.0 if n == 0 else float(min(1.0, 2 * sps.binom.cdf(min(n01, n10), n, 0.5)))

# ------------------------------------------------------------------ P3 batch-level machinery
def batch_acc(p, model):
    g = p[p.model == model]; return g.groupby(["grade", "batch_id", "seed"]).ok.mean().groupby(level=[0, 1]).mean().sort_index()   # seeds averaged first
def fold_acc(p, model):
    g = p[p.model == model]; return g.groupby(["fold", "seed"]).ok.mean().groupby(level=0).mean().sort_index()
def strat_idx(rng, n=NBOOT):
    """indices into the 25 batches ordered (grade, batch): resample the 5 batches within each grade, n draws"""
    idx = np.empty((n, 25), int)
    for g in range(5): idx[:, g * 5:(g + 1) * 5] = g * 5 + rng.integers(0, 5, (n, 5))
    return idx
def plain_idx(rng, n=NBOOT): return rng.integers(0, 25, (n, 25))
def fold_idx(rng, n=NBOOT):
    """resample whole folds (5 draws of fold k with replacement); fold k carries the batch k of every grade (5 batches)"""
    f = rng.integers(0, 5, (n, 5)); idx = np.empty((n, 25), int)
    for j in range(5):
        for g in range(5): idx[:, j * 5 + g] = g * 5 + f[:, j]
    return idx
def sign_flip_p(d):
    d = np.asarray(d, float); k = len(d); obs = abs(d.mean()); cnt = 0
    for s in itertools.product([-1, 1], repeat=k): cnt += abs((np.array(s) * d).mean()) >= obs - 1e-12
    return cnt / 2 ** k

def p3_tables(p, tag="P3"):
    rng = np.random.default_rng(0); idx = plain_idx(rng); idx_s = strat_idx(rng); idx_f = fold_idx(rng); models = [m for m in DEEP5 + HC4 if m in set(p.model)]
    U = {m: batch_acc(p, m).values for m in models}; F = {m: fold_acc(p, m).values for m in models}
    rows = []
    for m in models:
        bs, bss, bsf = (U[m][i].mean(1) for i in (idx, idx_s, idx_f)); rows.append(dict(protocol=tag, model=m, acc=U[m].mean(), ci_lo=np.quantile(bs, .025), ci_hi=np.quantile(bs, .975),
                                                 strat_lo=np.quantile(bss, .025), strat_hi=np.quantile(bss, .975), foldclu_lo=np.quantile(bsf, .025), foldclu_hi=np.quantile(bsf, .975),
                                                 fold_min=F[m].min(), fold_max=F[m].max(), n_batches=len(U[m])))
    T = pd.DataFrame(rows)
    pr = []
    for a, b in FAMILY:
        if a not in U or b not in U: continue
        d = U[a] - U[b]; bs = (U[a][idx] - U[b][idx]).mean(1); bss = (U[a][idx_s] - U[b][idx_s]).mean(1); bsf = (U[a][idx_f] - U[b][idx_f]).mean(1); fd = F[a] - F[b]
        pr.append(dict(protocol=tag, A=a, B=b, diff=d.mean(), ci_lo=np.quantile(bs, .025), ci_hi=np.quantile(bs, .975), ci90_lo=np.quantile(bs, .05), ci90_hi=np.quantile(bs, .95),
                       strat_lo=np.quantile(bss, .025), strat_hi=np.quantile(bss, .975), foldclu_lo=np.quantile(bsf, .025), foldclu_hi=np.quantile(bsf, .975),
                       p_wilcoxon_25batches=float(sps.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0, p_signflip_5folds=sign_flip_p(fd)))
    PW = pd.DataFrame(pr)
    if len(PW): PW["p_wilcoxon_holm"] = holm(PW.p_wilcoxon_25batches); PW["p_signflip_holm"] = holm(PW.p_signflip_5folds)
    return T, PW, U

def p1_tables(p):
    models = [m for m in DEEP5 + HC4 if m in set(p.model)]; rows, pr = [], []
    for m in models:
        g = p[p.model == m]; s = g.groupby("seed").ok.mean(); s1 = g[g.seed == g.seed.min()]; n = len(s1); k = int(s1.ok.sum()); z = 1.96; ph = k / n
        c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n)
        rows.append(dict(protocol="P1", model=m, acc=s.mean(), sd_seeds=s.std(ddof=1) if len(s) > 1 else np.nan, wilson_lo_firstseed=c - h, wilson_hi_firstseed=c + h, n_test=n))
    for a, b in FAMILY:
        ga, gb = p[p.model == a], p[p.model == b]
        if ga.empty or gb.empty: continue
        seeds = sorted(set(ga.seed) & set(gb.seed)); ps = []; diffs = []
        for s in seeds:
            x, y = ga[ga.seed == s].sort_values("sample_id"), gb[gb.seed == s].sort_values("sample_id"); ps.append(mcnemar_exact(x.ok.values, y.ok.values)); diffs.append(x.ok.mean() - y.ok.mean())
        ra, rb = ga.groupby("sample_id").ok.mean(), gb.groupby("sample_id").ok.mean(); d = (ra - rb.loc[ra.index]).values
        rng = np.random.default_rng(0); bs = rng.choice(d, (NBOOT, len(d))).mean(1)
        pr.append(dict(protocol="P1", A=a, B=b, diff=float(np.mean(diffs)), ci_lo=np.quantile(bs, .025), ci_hi=np.quantile(bs, .975), ci90_lo=np.quantile(bs, .05), ci90_hi=np.quantile(bs, .95),
                       p_mcnemar_max_over_seeds=max(ps), p_mcnemar_min_over_seeds=min(ps)))
    PW = pd.DataFrame(pr)
    if len(PW): PW["p_holm"] = holm(PW.p_mcnemar_max_over_seeds)
    return pd.DataFrame(rows), PW

def p2_tables(p):
    models = [m for m in DEEP5 + HC4 if m in set(p.model)]; rows, pr = [], []
    F = {m: p[(p.model == m) & (p.seed == p[p.model == m].seed.min())].groupby("fold").ok.mean().sort_index() for m in models}
    for m in models:
        f = F[m].values; rows.append(dict(protocol="P2", model=m, acc=f.mean(), sd_folds=f.std(ddof=1), n_folds=len(f)))
    tcrit = sps.t.ppf(.975, 9); t90 = sps.t.ppf(.95, 9)
    for a, b in FAMILY:
        if a not in F or b not in F: continue
        common = F[a].index.intersection(F[b].index); d = (F[a].loc[common] - F[b].loc[common]).values; k = len(d)
        if k < 2: continue
        se = np.sqrt((1 / k + 100 / 360) * d.var(ddof=1)); t = d.mean() / se if se > 0 else np.nan
        pr.append(dict(protocol="P2", A=a, B=b, diff=d.mean(), ci_lo=d.mean() - tcrit * se, ci_hi=d.mean() + tcrit * se, ci90_lo=d.mean() - t90 * se, ci90_hi=d.mean() + t90 * se,
                       p_nadeau_bengio=float(2 * sps.t.sf(abs(t), k - 1)) if se > 0 else 1.0))
    PW = pd.DataFrame(pr)
    if len(PW): PW["p_holm"] = holm(PW.p_nadeau_bengio)
    return pd.DataFrame(rows), PW

# ------------------------------------------------------------------ metrics (per class first, then macro) on pooled predictions
def metrics_table(p, tag):
    rows = []
    for m, g in p.groupby("model"):
        per_seed = []
        for s, h in g.groupby("seed"):
            pr, rc, f1, _ = precision_recall_fscore_support(h.y_true, h.y_pred, labels=list(range(5)), average=None, zero_division=0)   # zero-denominator -> 0
            per_seed.append(dict(acc=(h.y_true == h.y_pred).mean(), prec=pr.mean(), rec=rc.mean(), f1=f1.mean()))
        r = pd.DataFrame(per_seed).mean(); rows.append(dict(protocol=tag, model=m, accuracy=r.acc, macro_precision=r.prec, macro_recall=r.rec, macro_f1=r.f1))
    return pd.DataFrame(rows)

if __name__ == "__main__":
    out = {}
    T1, W1 = p1_tables(P1); T2, W2 = p2_tables(P2); T3, W3, U3 = p3_tables(P3)
    pd.concat([T1, T2, T3]).to_csv(f"{OUT}/table4_main.csv", index=False)
    pd.concat([W1, W2, W3]).to_csv(f"{OUT}/pairwise_family.csv", index=False)
    pd.concat([metrics_table(P1, "P1"), metrics_table(P2, "P2"), metrics_table(P3, "P3")]).to_csv(f"{OUT}/metrics_macro.csv", index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    print("P1\n", T1.round(3).to_string(), "\nP2\n", T2.round(3).to_string(), "\nP3\n", T3.round(3).to_string())
    print("\nPAIRWISE P1\n", W1.round(4).to_string(), "\nPAIRWISE P2\n", W2.round(4).to_string(), "\nPAIRWISE P3\n", W3.round(4).to_string())
    # segment ablation with uncertainty
    seg = []
    for sname in ["0_200", "200_500", "500_700"]:
        for prot, tag in [("original", "P1"), ("batch_grouped", "P3")]:
            p = load(f"seg_{sname}_{prot}")
            if p.empty: continue
            for m in p.model.unique():
                g = p[p.model == m]
                if tag == "P1": s = g.groupby("seed").ok.mean(); seg.append(dict(segment=sname.replace("_", "-"), protocol=tag, model=m, acc=s.mean(), sd_seeds=s.std(ddof=1), lo=np.nan, hi=np.nan))
                else:
                    u = batch_acc(p, m).values; rng = np.random.default_rng(0); b = u[strat_idx(rng)].mean(1)
                    seg.append(dict(segment=sname.replace("_", "-"), protocol=tag, model=m, acc=u.mean(), sd_seeds=g.groupby("seed").ok.mean().std(ddof=1), lo=np.quantile(b, .025), hi=np.quantile(b, .975)))
    pd.DataFrame(seg).to_csv(f"{OUT}/segment_uncertainty.csv", index=False); print("\nSEGMENT\n", pd.DataFrame(seg).round(3).to_string())
    open(f"{OUT}/METHODS.md", "w").write(__doc__)
'''
SRC["stats_extra.py"] = r'''"""Tables for the additional controls (E1-E4).  Run after the experiments in extra_exp.py finished (partial results are handled)."""
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(__file__))
import stats_v2 as S
from stats_v2 import load, batch_acc, plain_idx, NBOOT, RX, OUT, DEEP5, HC4

def acc_by(p, key):                      # mean over seeds of accuracy, then over splits
    return p.groupby(["model", "seed", key]).ok.mean().groupby(["model", key]).mean()
def split_sd(p, key="fold"):
    a = acc_by(p, key); return a.groupby("model").agg(["mean", "std", "count"])

def main():
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    res = {}
    # ---------------- E1: matched training data under P1
    d437 = load("e1_deep437", base=RX); h375 = load("e1_hc375", base=RX)
    if len(d437) and len(h375):
        rows = []
        for m in DEEP5:
            a = S.P1[S.P1.model == m].groupby("seed").ok.mean(); b = d437[d437.model == m].groupby("seed").ok.mean()
            rows.append(dict(model=m, kind="deep", acc_375_valsplit=a.mean(), sd_375=a.std(ddof=1), acc_437_fixedepochs=b.mean(), sd_437=b.std(ddof=1)))
        for m in HC4:
            a = S.P1[S.P1.model == m].groupby("seed").ok.mean(); b = h375[h375.model == m].groupby("seed").ok.mean()
            rows.append(dict(model=m, kind="handcrafted", acc_437_orig=a.mean(), sd_437=a.std(ddof=1), acc_375_matched=b.mean(), sd_375=b.std(ddof=1)))
        res["E1"] = pd.DataFrame(rows); res["E1"].to_csv(f"{OUT}/E1_matched_P1.csv", index=False); print("E1\n", res["E1"].round(3).to_string())
    # ---------------- E2: recording-level random control with the P3 budget
    e2 = pd.concat([load("e2_random_deep", base=RX), load("e2_random_hc", base=RX), load("e3_random", base=RX), load("e4_random", base=RX)], ignore_index=True)
    p3 = pd.concat([S.P3, load("e3_p3", base=RX), load("e4_p3", base=RX)], ignore_index=True)
    p1 = pd.concat([S.P1, load("e3_p1", base=RX), load("e4_p1", base=RX)], ignore_index=True)
    p2 = pd.concat([S.P2, load("e4_p2", base=RX)], ignore_index=True)
    if len(e2):
        rng = np.random.default_rng(0); idx = plain_idx(rng); rows = []
        e2s = split_sd(e2)
        for m in e2s.index:
            if m not in set(p3.model): continue
            u3 = batch_acc(p3, m).values; b3 = u3[idx].mean(1)
            sp = acc_by(e2, "fold").loc[m].values; be = sp[rng.integers(0, len(sp), (NBOOT, len(sp)))].mean(1)
            diff = be - b3
            r = dict(model=m, E2_random300=sp.mean(), E2_sd_splits=sp.std(ddof=1), n_splits=len(sp), E2_lo=np.quantile(be, .025), E2_hi=np.quantile(be, .975),
                     P3=u3.mean(), P3_lo=np.quantile(b3, .025), P3_hi=np.quantile(b3, .975), drop_E2_minus_P3=sp.mean() - u3.mean(), drop_lo=np.quantile(diff, .025), drop_hi=np.quantile(diff, .975))
            for nm, pp, fk in [("P1", p1, "fold"), ("P2", p2, "fold")]:
                if m in set(pp.model): r[nm] = acc_by(pp, fk).loc[m].mean()
            r["size_effect_P2_minus_E2"] = r.get("P2", np.nan) - r["E2_random300"]; rows.append(r)
        res["E2"] = pd.DataFrame(rows); res["E2"].to_csv(f"{OUT}/E2_random_control.csv", index=False); print("\nE2\n", res["E2"].round(3).to_string())
    # ---------------- E3/E4 tables: all extra models under P1, P2, P3
    ex = ["flatten_reg04", "flatten_pool8", "gap_wide", "bigru_u32", "gru_u96", "tcn", "transformer"]
    if any(len(load(f"e3_p3", base=RX)) for _ in [0]) or len(load("e4_p3", base=RX)):
        rng = np.random.default_rng(0); idx = plain_idx(rng); rows = []
        for m in ex:
            r = dict(model=m)
            for tag, pp in [("P1", p1), ("P2", p2), ("P3", p3), ("E2", e2)]:
                g = pp[pp.model == m] if len(pp) else pp
                if len(g) == 0: continue
                if tag == "P3":
                    u = batch_acc(pp, m).values; b = u[idx].mean(1); r.update(P3=u.mean(), P3_lo=np.quantile(b, .025), P3_hi=np.quantile(b, .975))
                else: a = acc_by(pp, "fold").loc[m]; r[tag] = a.mean(); r[tag + "_sd"] = a.std(ddof=1)
            rows.append(r)
        res["E34"] = pd.DataFrame(rows); res["E34"].to_csv(f"{OUT}/E3_E4_extra_models.csv", index=False); print("\nE3/E4\n", res["E34"].round(3).to_string())
        # paired contrasts of the controls against the originals under P3 (plain batch bootstrap) and P1 (McNemar max over seeds)
        pairs = [("cnn_flatten", "flatten_reg04"), ("cnn_gap", "gap_wide"), ("cnn_bigru", "bigru_u32"), ("cnn_gru", "gru_u96"), ("cnn_bigru", "gru_u96"),
                 ("cnn_bigru", "gap_wide"), ("cnn_bigru", "flatten_pool8"), ("cnn_bigru", "tcn"), ("cnn_bigru", "transformer"), ("cnn_gap", "tcn"), ("cnn_gap", "transformer"),
                 ("SVM-RBF", "tcn"), ("SVM-RBF", "transformer")]
        rows = []
        for a, b in pairs:
            if a in set(p3.model) and b in set(p3.model):
                ua, ub = batch_acc(p3, a).values, batch_acc(p3, b).values; bs = (ua[idx] - ub[idx]).mean(1)
                rows.append(dict(protocol="P3", A=a, B=b, diff=(ua - ub).mean(), lo=np.quantile(bs, .025), hi=np.quantile(bs, .975)))
        if rows: pd.DataFrame(rows).to_csv(f"{OUT}/E3_E4_contrasts_P3.csv", index=False); print("\nContrasts P3\n", pd.DataFrame(rows).round(3).to_string())
    for tag, pp in [("E2_all", e2)]:
        if len(pp): S.metrics_table(pp, tag).to_csv(f"{OUT}/metrics_{tag}.csv", index=False)
    ep = []
    for d in ["e1_deep437", "e2_random_deep", "e3_p1", "e3_p3", "e3_random", "e4_p1", "e4_p2", "e4_p3", "e4_random"]:
        f = f"{RX}/{d}/epochs.csv"
        if os.path.exists(f): ep.append(pd.read_csv(f, header=None, names=["model", "seed", "fold", "epochs", "trainable", "seconds"]).assign(exp=d))
    if ep: pd.concat(ep).to_csv(f"{OUT}/epochs_params_all.csv", index=False)

if __name__ == "__main__": main()
'''
for k, v in SRC.items(): open(f"{CODE}/{k}", "w").write(v)
ENV = dict(os.environ, ENOSE_NPZ=NPZ, ENOSE_OUT=f"{ROOT}/runs_extra", ENOSE_ROOT=ROOT, TF_THREADS="2")
def sh(*a):
    r = subprocess.run([sys.executable, *a], cwd=ROOT, env=ENV)
    if r.returncode: raise RuntimeError(f"failed: {a}")
EXTRA_JOBS = ["e1_hc", "e1_deep", "e2_hc", "e2_deep", "e3_p1", "e3_p3", "e3_e2", "e4_p1", "e4_p2", "e4_p3", "e4_e2"]
if RUN["extra"] and not QUICK:
    for j in EXTRA_JOBS: print("== extra", j); sh(f"{CODE}/extra_exp.py", j)         # resumable: finished (model, seed, fold) rows are skipped

== extra e1_hc
== extra e1_deep
== extra e2_hc
== extra e2_deep
== extra e3_p1
== extra e3_p3
== extra e3_e2
== extra e4_p1
== extra e4_p2


In [2]:
# %% 12. Revised statistics and Tables 4-7 (tables -> WORK/results/extra/stats_v2)
if RUN["stats_v2"] and not QUICK:
    sh(f"{CODE}/stats_v2.py"); sh(f"{CODE}/stats_extra.py"); print("tables in", f"{ROOT}/extra/stats_v2")

In [3]:
# %% 13. Package results (Colab: download)
import shutil
shutil.make_archive(f"{WORK}/results_enose", "zip", f"{WORK}/results"); print("zipped:", f"{WORK}/results_enose.zip")
try:
    from google.colab import files; files.download(f"{WORK}/results_enose.zip")
except ImportError:
    pass